In [ ]:
# import required libraries
using MultivariateStats
using Flux
using Bloqade
using Colors
using ProgressBars
using Plots
using CSV
using DataFrames
using Statistics
using StatsBase
     

In [ ]:
laser_data  = DataFrame(CSV.File("data/Laser.csv"))[!, "laser"]/255;
Plots.plot(laser_data[1:2000], label="")
xlabel!("times")
ylabel!("laser intensity")
     

In [ ]:
lnw=100
acc=pacf(laser_data, collect(1:lnw); method=:regression);
Plots.plot(collect(1:lnw), acc, label="")
xlabel!("lag window")
ylabel!("partial autocorrelation function")

In [ ]:
window=10;
train_data = laser_data[1:1400]
test_data=laser_data[1401:2000]
ln_train=length(train_data)
ln_test=length(test_data)
train_features=hcat([train_data[i:(i+window-1)] for i in 1:ln_train-window]...)
train_outcomes=hcat(train_data[window+1:end]...)
test_features=hcat([test_data[i:(i+window-1)] for i in 1:ln_test-window]...)
test_outcomes=hcat(test_data[window+1:end]...);
     

In [ ]:
# Define the DetuningLayer struct
Base.@kwdef struct DetuningLayer
    atoms # atom positions
    readouts # readout observables
    Ω::Float64 # Rabi frequency
    t_start::Float64 # evolution starting time
    t_end::Float64 # evolution ending time
    step::Float64  # window time step
    rate::Float64  # measurement rate per step
    reg::AbstractArrayReg # quantum state storage
end

In [ ]:
# The function implementing local detuning encoding of the feature x, identical to the MNIST notebook.
# For more details on simulation, please refer to Bloqade.jl and Yao.jl
function apply_layer(layer::DetuningLayer, x::Vector{<:Real})
    # define Rydberg hamiltonian, detunings parameterized in terms of feature vector values (x)  
     
    
    # At the start of the simulation, all atoms of the system are in ground states
    reg = layer.reg
    set_zero_state!(reg) 
    
    t_start = layer.t_start
    t_end = layer.t_end
    t_step = layer.step
    t_rate= layer.rate
    
    # initialize output vector
    steps = floor(Int, (t_end - t_start) / t_step * t_rate)
    out = zeros(steps * length(layer.readouts))
    h = rydberg_h(layer.atoms; Δ = x, Ω = layer.Ω)
    # Numerically simulate the quantum evolution with Krylov methods and store the readouts
    prob =  KrylovEvolution(reg, layer.t_start:(layer.step/layer.rate):layer.t_end, h)
    i=1
    for (step, reg, _) in prob # step through the state at each time step 
        step == 1 && continue # ignore first time step, this is just the initial state
        for op in layer.readouts
            out[i] = real(expect(op, reg)) # store the expectation of each operator for the given state in the output vector 
            i+=1
        end
    end
    return out
end
     

In [ ]:
function apply_layer_gl(layer::DetuningLayer, x::Vector{<:Real}, nshots=0)
    # define Rydberg hamiltonian, detunings parameterized in terms of feature vector values (x)  
     
    # At the start of the simulation, all atoms of the system are in ground states
    reg = layer.reg
    set_zero_state!(reg) 
    
    t_start = layer.t_start
    t_end = layer.t_end
    t_step = layer.step
    t_rate= layer.rate
    rabi_ramp=0.05
    # initialize output vector
    steps = floor(Int, (t_end - t_start) / t_step * t_rate)
    out = zeros(steps * length(layer.readouts))
    rabi = piecewise_linear(
        clocks = [0.0, rabi_ramp, layer.t_end-rabi_ramp, layer.t_end], 
        values=[0.0, layer.Ω, layer.Ω, 0.0])
    
    #piecewise linear encoding of the time-window feature vector (note, the last timepoint is repeated)
    Δpc = piecewise_linear(clocks = collect(t_start:t_step:t_end), values = push!(x,x[end])); 
    ϕ = piecewise_constant(clocks = [0, t_end] , values = [0])
    h = rydberg_h(layer.atoms; Δ = Δpc, Ω = rabi, ϕ=ϕ)
    # Numerically simulate the quantum evolution, we switch to ODE solver instead of the Krylov evolution as we no longer have time-constant pulses
    i = 1
    prob =  SchrodingerProblem(reg, layer.t_end, h, adaptive = true)
    integrator = init(prob, Vern8())
    for _ in TimeChoiceIterator(integrator, collect((t_start+t_step):t_step:t_end))
        prob.reg # state at selected time
        for op in layer.readouts
            if nshots > 0
                ex = sum(measure(op, reg; nshots)) / nshots
                out[i]=real(ex)
            else
                out[i] = real(expect(op, copy(prob.reg))) # store the expectation of each operator for the given state in the output vector
            end 
            i+=1
        end
    end
    return out
end
     
apply_layer_gl (generic function with 2 methods)



In [ ]:
# implement functions that apply a `DetuningLayer` to a matrix containing scaled detunings for each image
# one can additionaly choose the encoding time to use
function apply_layer(layer::DetuningLayer, x::Matrix{<:Real}; time_encoding=true)
    iter = SHOW_PROGRESS_BAR ? ProgressBar(1:size(x, 2)) : 1:size(x, 2)
    if time_encoding
        outs = [apply_layer_gl(layer, x[:, i][:]) for i in iter]
    else
        outs = [apply_layer(layer, x[:, i][:]) for i in iter]
    end
    return hcat(outs...)
end

In [ ]:
function train(xs_train, ys_train, xs_test, ys_test; 
    regularization::Float64 = 0.0, nepochs::Int = 100, batchsize::Int = 100, 
    opt = Flux.Adam(0.001), verbose::Bool=false, nonlinear::Bool=false)
    
    model = Dense(length(xs_train[:, 1]), 1)

    if nonlinear #additional feature - we can compare our results to 4-layer neural network
        model = Chain(
            Dense(length(xs_train[:, 1]), 100, relu),
            Dense(100, 100, relu),
            Dense(100, 1)
        )
    end

    loader = Flux.DataLoader((data = xs_train, label = ys_train); batchsize, shuffle=true);
    ps = Flux.params(model)

    verbose && println("Training...")
    losses = zeros(nepochs)
    losses_test = zeros(nepochs)
    for epoch in (verbose ? ProgressBar(1:nepochs) : 1:nepochs)
    l = 10000.0
    for (x, y) in loader
        grads = Flux.gradient(ps) do
            ŷ = model(x)
            if iszero(regularization)
                l = Flux.mse(ŷ, y, agg=sum)/sum((y .- mean(y)).^2)
            else
                l = Flux.mse(ŷ, y, agg=sum)/sum((y .- mean(y)).^2) + regularization * sum(sum(abs, p) for p in ps)
            end
        end
        Flux.update!(opt, ps, grads)
    end
    losses[epoch] = Flux.mse(model(xs_train), ys_train, agg=sum)/sum((ys_train .- mean(ys_train)).^2)
    losses_test[epoch] = Flux.mse(model(xs_test), ys_test, agg=sum)/sum((ys_test .- mean(ys_test)).^2)
    end
    return losses, losses_test, model
end
     

In [ ]:
losses, losses_test, model =train(train_features, train_outcomes, test_features, test_outcomes; 
regularization = 0.000, nepochs = 3000, batchsize = 1000, 
opt = Flux.Adam(0.01), verbose=false, nonlinear=false)

In [ ]:
println("Train NMSE linear= ",losses[end])
println("Test NMSE linear = ",losses_test[end])
Plots.plot(losses, label="Train NMSE")
Plots.plot!(losses_test, label="Test NMSE")
ylims!(0.,1.0)
xlabel!("epochs")
ylabel!("loss")
     


In [ ]:
Plots.plot(collect(50:150),test_outcomes[50:150], label="outcomes")
Plots.plot!(collect(50:150),model(test_features)[50:150], label="linear model predictions")
xlabel!("time")
ylabel!("laser intensity")

In [ ]:
losses, losses_test, model =train(train_features, train_outcomes, test_features, test_outcomes; 
regularization = 0.000, nepochs = 5000, batchsize = 1000, 
opt = Flux.Adam(0.004), verbose=false, nonlinear=true)
     

In [ ]:
println("Train NMSE NN= ",losses[end])
println("Test NMSE NN = ",losses_test[end])
Plots.plot(losses, label="Train NMSE")
Plots.plot!(losses_test, label="Test NMSE")
ylims!(0.,0.1)
xlabel!("epochs")
ylabel!("loss")
     


In [ ]:

Plots.plot(collect(50:150),test_outcomes[50:150], label="outcomes")
Plots.plot!(collect(50:150),model(test_features)[50:150], label="NN predictions")
xlabel!("time")
ylabel!("laser intensity")

In [ ]:
# Generate atom positions for the toy model
d = 10
nsites = 10
atoms = generate_sites(ChainLattice(), nsites; scale = d); # put atoms in a chain with 10 micron spacing
# create all single site Zᵢ and correlator ZᵢZⱼ readouts 
readouts = AbstractBlock[put(nsites, i => Z) for i in 1:nsites]
for i in 1:nsites
    for j in i+1:nsites
        push!(readouts, chain(put(nsites, i => Z), put(nsites, j => Z)))
    end
end

# build preprocessing layer 
pre_layer = DetuningLayer(;
    atoms, 
    readouts, 
    Ω = 2π, 
    t_start = 0.0, 
    t_end = 4.0, 
    step = 0.5,
    rate=1,
    reg = zero_state(nsites)
);

Δ_max = 6.0
xs = (train_features .- mean(train_features)) * Δ_max;
ys = (test_features .- mean(train_features)) * Δ_max;
     

In [ ]:
# uncomment the next line to see progress bar
SHOW_PROGRESS_BAR = false 
embeddings = apply_layer(pre_layer, xs, time_encoding=false)
test_embeddings = apply_layer(pre_layer, ys, time_encoding=false)

In [ ]:
losses, losses_test, model =train(embeddings, train_outcomes, test_embeddings, test_outcomes; 
regularization = 0.0001, nepochs = 5000, batchsize = 1000, 
opt = Flux.Adam(0.004), verbose=false, nonlinear=false)

In [ ]:
println("Train NMSE QRC= ",losses[end])
println("Test NMSE QRC = ",losses_test[end])
Plots.plot(losses, label="Train NMSE")
Plots.plot!(losses_test, label="Test NMSE")
ylims!(0.,0.1)
xlabel!("epochs")
ylabel!("loss")

In [ ]:
Plots.plot(collect(50:150),test_outcomes[50:150], label="outcomes")
Plots.plot!(collect(50:150),model(test_embeddings)[50:150], label="(local detuning) QRC model predictions")
xlabel!("time")
ylabel!("laser intensity")

In [ ]:
# irregular atom distances for the toy model, distances fixed for consistency
nsites = 10
rat=[8.3, 8.2, 8.2, 8.2, 8.1, 8.8, 8.2, 8.8, 8.8]
atoms=AtomList([(0.0, sum(rat[1:i])) for i in 0:(nsites-1)])
# create all single site Zᵢ and correlator ZᵢZⱼ readouts 
readouts = AbstractBlock[put(nsites, i => Z) for i in 1:nsites]
for i in 1:nsites
    for j in i+1:nsites
        push!(readouts, chain(put(nsites, i => Z), put(nsites, j => Z)))
    end
end

# build preprocessing layer 
pre_layer = DetuningLayer(;
    atoms, 
    readouts, 
    Ω = 2π*1.5, 
    t_start = 0.0, 
    t_end = 3.5, 
    step = 3.5/window,
    rate=1,
    reg = zero_state(nsites)
);

Δ_max = 9.0
xs = (train_features .- mean(train_features)) * Δ_max;
ys = (test_features .- mean(train_features)) * Δ_max;
     

In [ ]:
 #uncomment the next line to see progress bar
SHOW_PROGRESS_BAR = false 
embeddings = apply_layer(pre_layer, xs, time_encoding=true)
test_embeddings = apply_layer(pre_layer, ys, time_encoding=true)

In [ ]:
losses, losses_test, model =train(embeddings, train_outcomes, test_embeddings, test_outcomes; 
regularization = 0.0001, nepochs = 5000, batchsize = 1000, 
opt = Flux.Adam(0.004), verbose=false, nonlinear=false)
     

In [ ]:
println("Train NMSE QRC= ",losses[end])
println("Test NMSE QRC = ",losses_test[end])
Plots.plot(losses, label="Train NMSE")
Plots.plot!(losses_test, label="Test NMSE")
ylims!(0.,0.1)
xlabel!("epochs")
ylabel!("loss")
     

In [ ]:
Plots.plot(collect(50:150),test_outcomes[50:150], label="outcomes")
Plots.plot!(collect(50:150),model(test_embeddings)[50:150], label="(global detuning) QRC model predictions")
xlabel!("time")
ylabel!("laser intensity")
     